In [2]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))

In [3]:
from pathlib import Path
import json
import numpy as np

from models.backbone.modules.dataset import ELPVDataset


# ============================================================
# 1. Load the TRAIN dataset
# ============================================================

train_ds = ELPVDataset(
    split="train"
)

# Get normalization statistics calculated ONLY from train
train_mean, train_std = train_ds.computed_stats


# ============================================================
# 2. Load VALIDATION and TEST datasets
#    using the TRAIN normalization statistics
# ============================================================

val_ds = ELPVDataset(
    split="val",
    normalize_stats=(train_mean, train_std)
)

test_ds = ELPVDataset(
    split="test",
    normalize_stats=(train_mean, train_std)
)


# ============================================================
# 3. Dataset sizes
# ============================================================

print("===== DATASET SIZES =====")

print(f"Train:       {len(train_ds)}")
print(f"Validation:  {len(val_ds)}")
print(f"Test:        {len(test_ds)}")

print(
    f"Total:       "
    f"{len(train_ds) + len(val_ds) + len(test_ds)}"
)


# ============================================================
# 4. Normalization statistics
# ============================================================

print("\n===== NORMALIZATION =====")

print(f"Train-only mean: {train_mean:.6f}")
print(f"Train-only std:  {train_std:.6f}")


# ============================================================
# 5. Locate split_manifest.json
# ============================================================

project_root = Path.cwd()

if not (project_root / "split_manifest.json").exists():
    project_root = project_root.parent

manifest_path = project_root / "split_manifest.json"

if not manifest_path.exists():
    raise FileNotFoundError(
        f"split_manifest.json not found.\n"
        f"Expected location: {manifest_path}"
    )


# ============================================================
# 6. Load locked split manifest
# ============================================================

with open(manifest_path, "r") as f:
    manifest = json.load(f)


train_idx = set(manifest["train_idx"])
val_idx = set(manifest["val_idx"])
test_idx = set(manifest["test_idx"])


# ============================================================
# 7. Check for overlap between splits
# ============================================================

print("\n===== SPLIT INTEGRITY =====")

assert len(train_idx & val_idx) == 0, \
    "ERROR: Train/Validation overlap detected!"

assert len(train_idx & test_idx) == 0, \
    "ERROR: Train/Test overlap detected!"

assert len(val_idx & test_idx) == 0, \
    "ERROR: Validation/Test overlap detected!"

print("No Train/Validation overlap: PASSED")
print("No Train/Test overlap:       PASSED")
print("No Validation/Test overlap:  PASSED")


# ============================================================
# 8. Check that all 2624 ELPV samples are included
# ============================================================

all_indices = train_idx | val_idx | test_idx

assert len(all_indices) == 2624, \
    f"ERROR: Expected 2624 samples, found {len(all_indices)}"

print("All 2624 samples accounted for: PASSED")


# ============================================================
# 9. Verify class distribution
# ============================================================

print("\n===== CLASS DISTRIBUTION =====")

for name, ds in [
    ("Train", train_ds),
    ("Validation", val_ds),
    ("Test", test_ds)
]:

    counts = np.bincount(
        ds.labels,
        minlength=4
    )

    print(f"\n{name}:")

    for i, count in enumerate(counts):
        print(
            f"  {ELPVDataset.CLASS_NAMES[i]}: {count}"
        )


# ============================================================
# 10. Verify one actual dataset sample
# ============================================================

print("\n===== SAMPLE CHECK =====")

image, label = train_ds[0]

print("Image type:", type(image))
print("Image shape:", image.shape)
print("Image dtype:", image.dtype)
print("Label:", label.item())
print(
    "Label name:",
    ELPVDataset.CLASS_NAMES[label.item()]
)


# ============================================================
# FINAL
# ============================================================

print("\n===== DATASET VERIFICATION COMPLETE =====")
print("Permanent ELPV data pipeline: PASSED")

===== DATASET SIZES =====
Train:       1836
Validation:  394
Test:        394
Total:       2624

===== NORMALIZATION =====
Train-only mean: 0.598619
Train-only std:  0.160880

===== SPLIT INTEGRITY =====
No Train/Validation overlap: PASSED
No Train/Test overlap:       PASSED
No Validation/Test overlap:  PASSED
All 2624 samples accounted for: PASSED

===== CLASS DISTRIBUTION =====

Train:
  Healthy: 1055
  Mild: 207
  Moderate: 74
  Severe: 500

Validation:
  Healthy: 226
  Mild: 44
  Moderate: 16
  Severe: 108

Test:
  Healthy: 227
  Mild: 44
  Moderate: 16
  Severe: 107

===== SAMPLE CHECK =====
Image type: <class 'torch.Tensor'>
Image shape: torch.Size([1, 300, 300])
Image dtype: torch.float32
Label: 3
Label name: Severe

===== DATASET VERIFICATION COMPLETE =====
Permanent ELPV data pipeline: PASSED
